# 01 — Annotation-aware crop & rotation

Runs `scripts/image_crop_augment.py` on `raw/export.zip` and produces **lossless
1280×1280 PNG crops** plus a COCO annotation file under `crops_1280/`.

### Parameters
| Flag | Value | Meaning |
|---|---|---|
| `--crop-w/--crop-h` | 1280 | tile size (no runtime resize) |
| `-n` | 1400 | target number of crops |
| `--min-visibility` | 10 | a bbox less than 10 % visible at a tile edge is dropped / blacked out |
| `--output-format` | png | lossless (required — no compression) |
| `--workers` | 16 | parallel processes (I/O bound) |
| `--seed` | 42 | reproducible |

The script blacks out sub-threshold partial objects, de-duplicates, and guarantees that
every annotation is covered (forced crops).

## 1. Paths

In [ ]:
import os, subprocess, sys, time

ROOT = "/home/jovyan/shared/s0598584"
SCRIPT  = os.path.join(ROOT, "scripts", "image_crop_augment.py")
IN_ZIP  = os.path.join(ROOT, "raw", "export.zip")
OUT_DIR = os.path.join(ROOT, "crops_1280")
for p in (SCRIPT, IN_ZIP):
    assert os.path.exists(p), f"missing: {p}"
print("script :", SCRIPT)
print("input  :", IN_ZIP)
print("output :", OUT_DIR)

## 2. Run the script (venv Python as a subprocess)

In [ ]:
cmd = [
    sys.executable, SCRIPT,
    "-i", IN_ZIP,
    "-o", OUT_DIR,
    "--crop-w", "1280", "--crop-h", "1280",
    "-n", "1400",
    "--min-visibility", "10",
    "--output-format", "png",
    "--workers", "16",
    "--seed", "42",
]
print("CMD:", " ".join(cmd))
t0 = time.time()
proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(proc.stdout[-6000:])
print(f"\n[exit code {proc.returncode}]  duration: {time.time()-t0:.1f}s")
assert proc.returncode == 0, "image_crop_augment.py failed"

## 3. Verify the result: count & label distribution

In [ ]:
import json, glob
from collections import Counter
import cv2

img_dir = os.path.join(OUT_DIR, "images")
ann_path = os.path.join(OUT_DIR, "annotations_coco.json")

pngs = glob.glob(os.path.join(img_dir, "*.png"))
print("PNG crops on disk:", len(pngs))

with open(ann_path) as f:
    cc = json.load(f)
print("COCO images     :", len(cc["images"]))
print("COCO annotations:", len(cc["annotations"]))

cat_by_id = {c["id"]: c["name"] for c in cc["categories"]}
dist = Counter(a["category_id"] for a in cc["annotations"])
tot = sum(dist.values())
print("\nLabel distribution after cropping:")
for cid, n in dist.most_common():
    print(f"  {cat_by_id[cid]:16s} {n:6d}  ({n/tot*100:5.1f}%)")

im = cv2.imread(pngs[0])
print("\nExample crop shape:", im.shape, "(expected 1280,1280,3)")
assert im.shape[:2] == (1280, 1280)
assert len(pngs) == len(cc["images"]) > 0

## ✅ Phase 1 done

Lossless 1280×1280 PNG crops + `crops_1280/annotations_coco.json`. **Continue with
`02_photometric_augment.ipynb`.**